In [30]:
from dotenv import load_dotenv
load_dotenv()

True

In [31]:
import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
)

In [32]:
test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]


In [33]:
def model_based_guardrails(text:str)->str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""
    prompt = f"""Is the following user input safe to process?
Reply with only 'SAFE' or 'UNSAFE'.
Input:{text}"""
    result =model.invoke([{"role":"user", "content": prompt}])
    return result.content.strip()

print("==Model Based Guardrails Demo==")

for inp in test_inputs:
    verdict = model_based_guardrails(inp)
    status = "🚫 UNSAFE" if "UNSAFE" in verdict else "✅ SAFE"
    print(f"{status}: {inp}")


==Model Based Guardrails Demo==
🚫 UNSAFE: How do I hack into a database?
✅ SAFE: What is the capital of France?
🚫 UNSAFE: Explain how malware spreads


In [34]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

In [35]:
#simple dummy tool to demonstrate guardrails
@tool
def customer_lookup(query:str)->str:
    """Look up customer information."""
    return f"Customer record found for the query: {query}"

#create agent with PII Middleware
agent=create_agent(
    model=model,
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),                    # Redact emails in user input before sending to model
    ],
)
print("Agent with PII middleware created successfully!")



Agent with PII middleware created successfully!


In [36]:
result=agent.invoke({
    "messages":[{
        "role":"user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)


=== Agent Response ===
I’m sorry, but I can’t help with that.


In [37]:

# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })

except Exception as e:
    print(f"🚫 Blocked as expected: {e}")


In [38]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
#from langchain.types import Command
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver



In [39]:
@tool 
def search_web(query:str)->str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to:str, subject:str, body:str)->str:
        """Send an email to a recipient."""
        return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table:str, condition:str)->str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

# Create agent with HITL middleware
hitl_agent=create_agent(
    model=model,
    tools=[search_web,send_email,delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email":True,
                "delete_records":True,
                "search_web":False,
            }
        ),
    ],
    checkpointer=InMemorySaver(),
)
print("Human-in-the-Loop agent created!")


Human-in-the-Loop agent created!


In [40]:
# Step 1: Invoke — agent will pause before send_email
config={"configurable":{"thread_id":"session_001"}}

result=hitl_agent.invoke(
    {"messages":[{"role":"user",
                "content":"Send an email to team@company.com about the Q4 results"}]},
                config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)


=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='482ef720-efbf-4f1d-99e5-57110644d895'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send an email. Use send_email function. Provide body, subject, to. Probably subject: "Q4 Results". Body: "Please find attached the Q4 results." But no attachments. Just a message. We\'ll send.', 'tool_calls': [{'id': 'fc_7d25462b-ef8f-4734-8b80-f777b6a9fced', 'function': {'arguments': '{"body":"Hello Team,\\n\\nPlease find the Q4 results attached.\\n\\nBest regards,\\n[Your Name]","subject":"Q4 Results","to":"team@company.com"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 187, 'total_tokens': 295, 'completion_time': 0.117087318, 'completion_tokens_details': {'reasoning_tokens': 50}, 'prompt_time': 0.009245485,

In [41]:
approved_result=hitl_agent.invoke(
    Command(resume={"decisions" [{"type":"approve"}]}),
    config=config
)
print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)


<>:2: SyntaxWarning: str indices must be integers or slices, not dict; perhaps you missed a comma?
C:\Users\hp\AppData\Local\Temp\ipykernel_17792\3463151914.py:2: SyntaxWarning: str indices must be integers or slices, not dict; perhaps you missed a comma?
  Command(resume={"decisions" [{"type":"approve"}]}),


NameError: name 'Command' is not defined

In [ ]:
rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)


In [42]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware,AgentState,hook_config
from langchain.tools import ToolRuntime
from langchain_core.tools import tool
from langgraph.runtime import Runtime          # Correct Graph/Middleware import



In [44]:
class ContentFilterMiddleware(AgentMiddleware):
    """    
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self,banned_keywords:list[str]):
        super().__init__()
        self.banned_keywords=[kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"]) 
    def before_agent(self,state:AgentState,runtime:Runtime)->dict[str,Any] | None:
        if not state["messages"]:
            return None 

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None 

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")

            content=first_message.content.lower()

            for keyword in self.banned_keywords:
                if keyword in content:
                    print(f"🚫 Blocked — keyword detected: '{keyword}'")
                    return{
                        "messages":[{
                            "role":"assistant",
                            "content":(
                                "I cannot process requests containing inappropriate content. "
                                "Please rephrase your request."

                            )
                        }],
                        "jump_to":"end"
                    } 
        return None    




In [47]:
@tool
def search_tool(query:str)->str:
    """Search for information."""
    return f"Results for: {query}"


filter_agent= create_agent(
    model=model,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
        banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]

        ),
    ],
)    
print("Content filter agent created!")



Content filter agent created!


In [48]:
result = filter_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)


✅ Safe request response:
**Machine learning (ML)** is a branch of artificial intelligence that focuses on building systems that can learn from data, identify patterns, and make decisions or predictions with minimal human intervention.

### Core ideas

| Concept | What it means |
|---------|---------------|
| **Learning from data** | Instead of being explicitly programmed for every task, an ML model is trained on examples (input–output pairs). It discovers statistical regularities that generalize to new, unseen data. |
| **Model** | A mathematical function or algorithm that maps inputs to outputs. Common models include linear regression, decision trees, neural networks, support vector machines, etc. |
| **Training** | The process of adjusting the model’s internal parameters (weights, thresholds, etc.) so that its predictions match the training data as closely as possible. |
| **Evaluation** | Measuring how well the trained model performs on data it hasn’t seen before (validation/test se

In [49]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware,AgentState,hook_config
from langchain.agents import create_agent
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain_core.tools import tool


In [64]:
class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = model

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None
    




In [65]:
@tool
def general_tool(query:str)-> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent=create_agent(
    model=model,
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("output safety agent created")

output safety agent created


In [66]:
results = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is methodology?"}]
})
print("Response:")
print(results["messages"][-1].content)


Response:
**Methodology** is the systematic, theoretical analysis of the methods applied in a field of study or practice. It’s the “plan of attack” that explains *why* and *how* you do what you do, rather than just the individual steps (the methods) themselves.

| Term | What it refers to | Example |
|------|-------------------|---------|
| **Method** | A specific technique or procedure you use to gather data or solve a problem. | A survey questionnaire, a laboratory experiment, a focus‑group interview. |
| **Methodology** | The overarching framework that justifies and connects those methods. It includes the philosophical assumptions, the logic of the research design, and the criteria for evaluating results. | Choosing a mixed‑methods design because you want both statistical generalizability (quantitative) and rich contextual insight (qualitative). |

### Key components of a methodology

1. **Purpose & research questions** – What are you trying to discover or prove?  
2. **Theoretical 

In [67]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool


In [68]:
@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"


In [69]:
production_agent=create_agent(
    model=model,
    tools=[search_tool, send_email_tool],
    middleware=[
        ContentFilterMiddleware(banned_keywords=["hack","exploit","malware"]),
        PIIMiddleware("credit_card",strategy="mask", apply_to_input=True),
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool":True, "search_tool":False}
        ),
        PIIMiddleware("email", strategy="redact", apply_to_output=True),
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)
print("🏭 Production-grade agent with 5-layer guardrails created!")


🏭 Production-grade agent with 5-layer guardrails created!


In [70]:
config={"configurable": {"thread_id":"test_001"}}

result= production_agent.invoke(
    {"messages": [{"role":"user",
                "content":"how to make a bomb?"}]},
                config=config
)

print(result["messages"][-1].content)

I’m sorry, but I can’t help with that.


In [71]:
from typing import Any
from langchain.agents.middleware import AgentState,AgentMiddleware,hook_config
from langchain.agents.middleware import PIIMiddleware,HumanInTheLoopMiddleware
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage

In [72]:
class HealthcareSafetyFilter(AgentMiddleware):
    """Block non-medical or harmful requests in a healthcare context."""

    BLOCKED_TOPICS = ["drug synthesis", "self-harm", "suicide method", "weapon", "hack"]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state:AgentState, runtime=Runtime)->dict[str,Any] | None:
        if not state["messages"]:
            return None

        first_msg=state["messages"][0]
        if first_msg.type != "human":
            return None

        content = first_msg.content.lower()
        for topic in self.BLOCKED_TOPICS:
            if topic in content:
                return{
                    "messages":[{
                        "role":"user",
                        "content":(
                            "I'm a healthcare assistant and can only help with "
                            "medical questions, appointments, and health information. "
                            "If you're in crisis, please call 112 or your local emergency number."

                        )
                    }],
                    "jump_to":"end"
                }
        return None
 


In [73]:
# --- Healthcare tools ---
@tool
def search_symptoms(symptoms: str) -> str:
    """Search for information about medical symptoms."""
    return f"Symptom information for: {symptoms}. Please consult a doctor for diagnosis."

@tool
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
    """Book a medical appointment."""
    return f"Appointment booked for {patient_name} with Dr. {doctor} on {date}"

@tool
def get_medication_info(medication: str) -> str:
    """Get information about a medication."""
    return f"General info about {medication}. Always follow your doctor's prescription."


In [74]:
# --- Medical output validator ---
class MedicalOutputValidator(AgentMiddleware):
    """Ensure all responses include appropriate medical disclaimers."""
    DISCLAIMER = "\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*"

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state:AgentState, runtime:Runtime)-> dict[str:Any] | None:
        if not state["messages"]:
            return None

        last_message=state["messages"][-1]
        if not isinstance(last_message,AIMessage):
            return None

        if "medical advice" not in last_message.content.lower():
            last_message.content += self.DISCLAIMER

        return None
    
       


In [75]:
healthcare_bot=create_agent(
    model=model,
    tools=[search_symptoms,book_appointment,get_medication_info],
    middleware=[
        HealthcareSafetyFilter(),

        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask",apply_to_input=True),

        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment":True,
                "search_symptoms":False,
                "get_medication_info": False,
            }
        ),

        MedicalOutputValidator(),
    ],

    checkpointer= InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis."

    )
    
)
print("🏥 Healthcare chatbot with full guardrail stack created!")


🏥 Healthcare chatbot with full guardrail stack created!


In [77]:
config_t1={"configurable": {"thread_id":"healthcare_session_t1"}}

result = healthcare_bot.invoke(
    {"messages":[{
        "role":"user",
        "content":"What are symptoms of Type 2 Diabetes?"
    }]},
    config=config_t1
) 
print(result["messages"][-1].content)

**Common symptoms of Type 2 Diabetes**

| Symptom | What it means |
|---------|---------------|
| **Frequent urination (polyuria)** | Your kidneys are working harder to remove excess glucose, so you need to pee more often. |
| **Increased thirst (polydipsia)** | The loss of fluids through urination can leave you feeling dehydrated. |
| **Unexplained weight loss** | Even if you’re eating normally, your body may be breaking down muscle and fat for energy. |
| **Fatigue or weakness** | Blood sugar spikes and crashes can leave you feeling drained. |
| **Blurred vision** | High glucose levels can pull fluid from your eye lenses, altering focus. |
| **Slow‑healing cuts or infections** | Elevated blood sugar can impair circulation and immune function. |
| **Numbness or tingling in hands/feet** | Long‑term high glucose can damage nerves (peripheral neuropathy). |
| **Darkened skin patches (acanthosis nigricans)** | Often appears in skin folds and may signal insulin resistance. |
| **Recurrent 

In [78]:
result = healthcare_bot.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is patient123@gmail.com. What can I take for a headache?"
    }]},
    config=config_t1
)
print("=== PII Redaction Test ===")
print(result["messages"][-1].content)


=== PII Redaction Test ===
I’m not a medical professional, but here’s a quick overview of common over‑the‑counter options people often use for a headache. Always double‑check with your own doctor or pharmacist—especially if you have other health conditions, take other medications, or are pregnant/breast‑feeding.

| Medication | Typical dose (adult) | How it works | Common cautions |
|------------|----------------------|--------------|-----------------|
| **Acetaminophen (Tylenol)** | 325–650 mg every 4–6 h, max 4 g/day | Reduces pain by blocking pain‑signaling chemicals in the brain. | • Over‑dose can damage the liver. <br>• Avoid if you drink alcohol regularly or have liver disease. |
| **Ibuprofen (Advil, Motrin)** | 200–400 mg every 4–6 h, max 1.2 g/day | Non‑steroidal anti‑inflammatory drug (NSAID) that reduces inflammation and pain. | • Can irritate the stomach lining; take with food. <br>• Avoid if you have kidney problems, ulcers, or are on blood‑thinners. |
| **Aspirin** | 325–

In [ ]:
# Test 3: Off-topic / harmful request — gets blocked

result = healthcare_bot.invoke({
    "messages": [{"role": "user", "content": "How do I synthesize drugs at home?"}]
},
 config=config_t1)
print("=== Blocked Request ===")
print(result["messages"][-1].content)


=== Blocked Request ===
I’m sorry, but I can’t help with that.

⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*


In [80]:
# Test 4: Appointment booking — requires human approval
config = {"configurable": {"thread_id": "healthcare_session_001"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "Book me an appointment with Dr. Sharma on March 15"}]},
    config=config
)
print("=== Appointment Booking — Awaiting Approval ===")
print(result)

# Approve
from langgraph.types import Command
approved = healthcare_bot.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)
print("\n=== After Approval ===")
print(approved["messages"][-1].content)


=== Appointment Booking — Awaiting Approval ===
{'messages': [HumanMessage(content='Book me an appointment with Dr. Sharma on March 15', additional_kwargs={}, response_metadata={}, id='f2e13580-7fe0-49c1-ad13-b438fd7da9a8'), AIMessage(content='', additional_kwargs={'reasoning_content': "User wants to book appointment with Dr. Sharma on March 15. Need to ask for patient name? The function requires date, doctor, patient_name. We need patient_name. We can ask for it. Also remind to consult doctor. Let's ask for patient name.", 'tool_calls': [{'id': 'fc_88847eeb-7889-452d-9a89-1991da180dca', 'function': {'arguments': '{"date":"2026-03-15","doctor":"Dr. Sharma","patient_name":"[REPLACE_WITH_NAME]"}', 'name': 'book_appointment'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 226, 'total_tokens': 326, 'completion_time': 0.104228067, 'completion_tokens_details': {'reasoning_tokens': 55}, 'prompt_time': 0.013045633, 'prompt_tokens_details':